<a href="https://colab.research.google.com/github/RajeevKambhatla/longevity-drug-repurposing/blob/main/18_benchmarks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import json, numpy as np, pandas as pd
BASE = '/content/drive/MyDrive/longevity_project'
kd = json.load(open(f'{BASE}/data/processed/known_drugs.json'))
final = pd.read_parquet(f'{BASE}/results/final_scores.parquet')
sigpca = pd.read_parquet(f'{BASE}/data/processed/signature_pca.parquet')
HALLMARKS = list(kd['by_hallmark'])
meta_cols = [f'{h}_meta' for h in HALLMARKS]

named = final[final['is_named']].copy()
named['overall'] = named[meta_cols].mean(axis=1).rank(pct=True) * 100
named['best_hallmark'] = named[meta_cols].max(axis=1).rank(pct=True) * 100
named['best_which'] = named[meta_cols].idxmax(axis=1).str.replace('_meta', '')
named['signal_strength'] = pd.Series(np.linalg.norm(sigpca.values, axis=1), index=sigpca.index).reindex(named.index).rank(pct=True) * 100
known = {d for k in kd['by_hallmark'].values() for d in k['helpful'] + k['harmful']}
print(len(named), 'named compounds')

Mounted at /content/drive
2966 named compounds


In [2]:
# Clear lifespan extension in the ITP (at least one sex)
ITP_POSITIVE = ['sirolimus', 'acarbose', 'alpha-estradiol', 'nordihydroguaiaretic-acid',
                'aspirin', 'canagliflozin', 'glycine', 'captopril']
# Clearly no lifespan extension in the ITP
ITP_NEGATIVE = ['resveratrol', 'simvastatin', 'curcumin', 'oxaloacetic-acid', 'methylene-blue',
                'dichloroacetic-acid', 'forskolin', 'fisetin', 'pioglitazone', 'mifepristone', 'methotrexate']
# Left out because results conflict between studies or analyses
ITP_EXCLUDED = ['metformin', 'enalapril', 'caffeic-acid-phenethyl-ester', 'green tea extract', '17-DMAG',
                'astaxanthin', 'meclizine', 'mitoglitazone', 'epicatechin', 'halofuginone', 'protandim']

def similar(name):
    key = name.split('-')[0][:6]
    return [n for n in named.index if key in n][:6]

for n in ITP_POSITIVE + ITP_NEGATIVE:
    print(f"{n:28} {'found' if n in named.index else 'NOT FOUND -> similar names: ' + str(similar(n))}")

sirolimus                    found
acarbose                     found
alpha-estradiol              found
nordihydroguaiaretic-acid    NOT FOUND -> similar names: []
aspirin                      found
canagliflozin                NOT FOUND -> similar names: []
glycine                      NOT FOUND -> similar names: []
captopril                    found
resveratrol                  found
simvastatin                  found
curcumin                     found
oxaloacetic-acid             NOT FOUND -> similar names: []
methylene-blue               found
dichloroacetic-acid          found
forskolin                    found
fisetin                      NOT FOUND -> similar names: ['fisetinidol']
pioglitazone                 found
mifepristone                 found
methotrexate                 found


In [3]:
def auc(pos, neg):
    p, n = np.asarray(pos)[:, None], np.asarray(neg)[None, :]
    return ((p > n).sum() + 0.5 * (p == n).sum()) / (p.size * n.size)

def benchmark(pos, neg, col, n_perm=10000):
    s = named.loc[pos + neg, col].values
    obs = auc(s[:len(pos)], s[len(pos):])
    rng = np.random.default_rng(0)
    null = [auc(x[:len(pos)], x[len(pos):]) for x in (rng.permutation(s) for _ in range(n_perm))]
    return obs, float(np.mean(np.array(null) >= obs))

results = {}
for label, drop_known in [('independent', True), ('all', False)]:
    pos = [d for d in ITP_POSITIVE if d in named.index and not (drop_known and d in known)]
    neg = [d for d in ITP_NEGATIVE if d in named.index and not (drop_known and d in known)]
    print(f'\n=== ITP, {label} ({len(pos)} extended lifespan vs {len(neg)} did not) ===')
    if len(pos) < 2 or len(neg) < 2:
        print('Too few compounds for this version.')
        continue
    for col in ['overall', 'best_hallmark', 'signal_strength']:
        a, p = benchmark(pos, neg, col)
        results[f'itp_{label}_{col}'] = {'auc': round(a, 3), 'p': round(p, 3), 'n_pos': len(pos), 'n_neg': len(neg)}
        print(f'{col:16} AUC {a:.2f}  (p = {p:.2f})')
    table = named.loc[pos + neg, ['overall', 'best_hallmark', 'best_which']].round(0)
    table['ITP result'] = ['extended lifespan'] * len(pos) + ['no effect'] * len(neg)
    table['in training?'] = ['yes' if d in known else 'no' for d in table.index]
    print(table.sort_values('overall', ascending=False).to_string())


=== ITP, independent (3 extended lifespan vs 6 did not) ===
overall          AUC 0.56  (p = 0.45)
best_hallmark    AUC 0.17  (p = 0.95)
signal_strength  AUC 0.28  (p = 0.87)
                     overall  best_hallmark        best_which         ITP result in training?
name                                                                                         
alpha-estradiol         94.0           34.0  nutrient_sensing  extended lifespan           no
pioglitazone            94.0           85.0        senescence          no effect           no
methotrexate            79.0           99.0      proteostasis          no effect           no
captopril               60.0           30.0         autophagy  extended lifespan           no
forskolin               40.0           54.0        senescence          no effect           no
methylene-blue          22.0           79.0         autophagy          no effect           no
mifepristone            14.0           59.0      proteostasis          no

In [4]:
drugage = set(kd['drugage'])
pool = named[~named.index.isin(known)].copy()
for col in ['overall', 'best_hallmark', 'signal_strength']:
    pool[col] = pool[col].rank(pct=True) * 100
is_da = pool.index.isin(drugage)
print(f'{is_da.sum()} DrugAge lifespan extenders vs {(~is_da).sum()} other named compounds (known drugs excluded)\n')

named_backup, named = named, pool
for col in ['overall', 'best_hallmark', 'signal_strength']:
    a, p = benchmark(list(pool.index[is_da]), list(pool.index[~is_da]), col, n_perm=2000)
    results[f'drugage_{col}'] = {'auc': round(a, 3), 'p': round(p, 4), 'n_pos': int(is_da.sum())}
    print(f'{col:16} AUC {a:.2f}  (p = {p:.3f})   average percentile of DrugAge compounds: {pool.loc[is_da, col].mean():.0f}')

print('\nBy hallmark (which aging process best picks out DrugAge compounds?):')
for h in HALLMARKS:
    print(f'{h:17} AUC {auc(pool.loc[is_da, f"{h}_meta"], pool.loc[~is_da, f"{h}_meta"]):.2f}')
named = named_backup

124 DrugAge lifespan extenders vs 2805 other named compounds (known drugs excluded)

overall          AUC 0.50  (p = 0.556)   average percentile of DrugAge compounds: 50
best_hallmark    AUC 0.55  (p = 0.024)   average percentile of DrugAge compounds: 55
signal_strength  AUC 0.56  (p = 0.007)   average percentile of DrugAge compounds: 56

By hallmark (which aging process best picks out DrugAge compounds?):
senescence        AUC 0.46
nutrient_sensing  AUC 0.55
mitochondria      AUC 0.48
autophagy         AUC 0.48
inflammation      AUC 0.47
proteostasis      AUC 0.54


In [5]:
json.dump({'itp_positive': ITP_POSITIVE, 'itp_negative': ITP_NEGATIVE, 'itp_excluded': ITP_EXCLUDED,
           'primary_score': 'overall (mean of six meta-model hallmark percentiles)', 'results': results},
          open(f'{BASE}/results/benchmarks.json', 'w'), indent=2)

r = results
itp = r.get('itp_independent_overall')
print(f"Benchmarked final scores on held-out lifespan data, using the mean of six hallmark scores as the pre-specified "
      f"primary score. ITP (excluding training drugs): " +
      (f"AUC {itp['auc']:.2f} (p = {itp['p']:.2f}; {itp['n_pos']} lifespan-extending vs {itp['n_neg']} failed compounds)"
       if itp else 'too few independent compounds to test') +
      f". DrugAge ({r['drugage_overall']['n_pos']} lifespan-extending compounds vs other named compounds): AUC "
      f"{r['drugage_overall']['auc']:.2f} (p = {r['drugage_overall']['p']:.3f}), compared with "
      f"{r['drugage_signal_strength']['auc']:.2f} for signature strength alone. Contested ITP compounds excluded and listed.")

Benchmarked final scores on held-out lifespan data, using the mean of six hallmark scores as the pre-specified primary score. ITP (excluding training drugs): AUC 0.56 (p = 0.45; 3 lifespan-extending vs 6 failed compounds). DrugAge (124 lifespan-extending compounds vs other named compounds): AUC 0.50 (p = 0.556), compared with 0.56 for signature strength alone. Contested ITP compounds excluded and listed.
